## Exercício 1

In [ ]:
%pip install pandas

In [1]:
import pandas as pd
# Importa a classe que criamos no arquivo vizinho
from database_connector import BancoDeDados

# 1. Instancia a classe do banco
db = BancoDeDados()

# 2. Seu código original de leitura do CSV local
path_csv = r"C:\Users\Aluno\Downloads\MICRODADOS.csv"
print("Lendo o arquivo CSV...")

df = pd.read_csv(
    path_csv, 
    sep=';', 
    encoding='LATIN1', 
    low_memory=False
)

# 3. Seu código original de processamento de nulos
null_counts = df.isnull().sum()
null_percentages = (df.isnull().sum() / len(df)) * 100

report_df = pd.DataFrame({
    'Coluna CSV': null_counts.index,
    'Qtd Nulos': null_counts.values,
    'Percentual Nulos (%)': null_percentages.values
})
report_df = report_df.sort_values(by='Percentual Nulos (%)', ascending=False).reset_index(drop=True)

# --- AQUI ENTRA A MÁGICA DO BANCO NA NUVEM ---

# Em vez de salvar só o CSV local, enviamos o relatório direto para a nuvem
db.salvar_dataframe(report_df, nome_tabela='relatorio_nulos_covid')

# Exibe o resultado na tela
print("\n--- Top 15 Colunas com mais valores nulos ---")
print(report_df.head(15).to_string(index=False))

Lendo o arquivo CSV...
Enviando dados para a tabela 'relatorio_nulos_covid'...
[ERRO] Falha ao salvar no banco: (psycopg2.OperationalError) connection to server at "localhost" (::1), port 5432 failed: Connection refused (0x0000274D/10061)
	Is the server running on that host and accepting TCP/IP connections?
connection to server at "localhost" (127.0.0.1), port 5432 failed: Connection refused (0x0000274D/10061)
	Is the server running on that host and accepting TCP/IP connections?

(Background on this error at: https://sqlalche.me/e/20/e3q8)

--- Top 15 Colunas com mais valores nulos ---
            Coluna CSV  Qtd Nulos  Percentual Nulos (%)
             DataObito    5182016             99.548479
   DataColetaSorologia    5105399             98.076638
DataColetaSorologiaIGG    5058719             97.179897
     DataColeta_RT_PCR    3585036             68.869892
 DataColetaTesteRapido    1945394             37.371752
      DataEncerramento     226559              4.352284
               

## Exercício 2

In [2]:
from database_connector import BancoDeDados

# 1. Instancia o conector do banco
db = BancoDeDados()

# 2. Executa o Passo 1 (Recriar o banco de dados)
banco_pronto = db.recriar_banco_dw()

# 3. Se o banco foi criado com sucesso, executa os Passos 2 e 3 (Script SQL)
if banco_pronto:
    SQL_FILE_PATH = r"sql\snowflake.sql"
    db.executar_script_snowflake(SQL_FILE_PATH)

Passo 1: Gerenciando o Banco de Dados no Servidor...
[ERRO CRÍTICO] Não foi possível recriar o banco: (psycopg2.OperationalError) connection to server at "localhost" (::1), port 5432 failed: Connection refused (0x0000274D/10061)
	Is the server running on that host and accepting TCP/IP connections?
connection to server at "localhost" (127.0.0.1), port 5432 failed: Connection refused (0x0000274D/10061)
	Is the server running on that host and accepting TCP/IP connections?

(Background on this error at: https://sqlalche.me/e/20/e3q8)


## Exercício 3

In [ ]:
from database_connector import BancoDeDados

# 1. Instancia o conector do banco
db = BancoDeDados()

# 2. Executa o Passo 1 (Recriar o banco de dados)
banco_pronto = db.recriar_banco_dw()

# 3. Se o banco foi criado com sucesso, executa os Passos 2 e 3 (Script SQL)
if banco_pronto:
    SQL_FILE_PATH = r"sql\snowflake.sql"
    db.executar_script_snowflake(SQL_FILE_PATH)

## Exercício 4

In [ ]:
from database_connector import BancoDeDados

# 1. Instancia o conector do banco (ele já conhece as credenciais)
db = BancoDeDados()

# 2. Define o caminho do script do Data Mart
SQL_MART_PATH = r"sql\mart_municipio_mes.sql"

# 3. Dispara todo o processo de criação e análise de performance
db.processar_e_testar_data_mart(SQL_MART_PATH)

## Exercício 5

## Exercício 6

### 1. Novo Esquema da Tabela dw.dim_municipio
A tabela ganha campos de controle de vigência (data_inicio, data_fim) e um indicador de registro ativo (flag_atual).

Além disso, a sk_municipio continua sendo a Chave Substituta (Surrogate Key) sequencial única, garantindo que o mesmo município possa ter várias linhas no banco (uma para cada período populacional).

"CREATE TABLE dw.dim_municipio (
    -- Chave substituta garante a unicidade de cada versão histórica
    sk_municipio SERIAL PRIMARY KEY, 
    
    -- Chave de negócio (Código IBGE ou Nome do Município)
    municipio VARCHAR(100) NOT NULL,
    uf CHAR(2) DEFAULT 'ES',
    fk_regiao INT NOT NULL REFERENCES dw.dim_regiao(sk_regiao),
    
    -- O novo atributo mutável no tempo
    populacao_municipio INT,
    
    -- Campos de controle do SCD Tipo 2
    data_inicio DATE NOT NULL,      -- Quando essa versão da população começou a valer
    data_fim DATE,                  -- Quando essa versão foi substituída (NULL se for a atual)
    flag_atual BOOLEAN NOT NULL     -- TRUE para a versão ativa, FALSE para versões antigas
);"

### 2. Fluxo de Funcionamento no ETL (Lógica de Atualização)

Quando o processo de ETL rodar e detectar que a população de um município mudou no arquivo original (ex: Vila Velha mudou de 500k para 520k habitantes):

    Fecha o registro antigo: O ETL localiza a linha atual daquele município (flag_atual = TRUE), altera a data_fim para a data de ontem (ex: 2026-06-14) e muda o flag_atual para FALSE.

    Abre o registro novo: O ETL insere uma linha novinha para o município com a população atualizada, definindo data_inicio = '2026-06-15', data_fim = NULL e flag_atual = TRUE.

### 3. Impacto nas Consultas (Fato -> Dimensão)

O maior benefício do SCD Tipo 2 é a preservação da foto histórica do momento exato do evento.

    Quando o processo de carga da fato_notificacao_covid rodar, o LEFT JOIN que busca a chave do local não vai olhar apenas o nome da cidade. Ele obrigatoriamente cruzará a data da notificação com o período de vigência da dimensão.

    "SELECT 
        m.sk_municipio
    FROM stg.notificacao_raw s
    JOIN dw.dim_municipio m 
    ON m.municipio = s.municipio 
    AND s.DataNotificacao::DATE BETWEEN m.data_inicio AND COALESCE(m.data_fim, '2026-12-31'::DATE);"

## Exercício 7

In [ ]:
import os
from sqlalchemy import create_engine

# Configurações de Conexão
USER = "postgres"
PASSWORD = "6794"
HOST = "localhost"
PORT = "5432"
SQL_PROCEDURE_PATH = r"sql\quality_gate_fato.sql"

URL_DW = f"postgresql+psycopg2://{USER}:{PASSWORD}@{HOST}:{PORT}/dw_covid"
engine_dw = create_engine(URL_DW)

print("Passo 1: Carregando o arquivo da Stored Procedure...")
if not os.path.exists(SQL_PROCEDURE_PATH):
    print(f"[ERRO] O arquivo {SQL_PROCEDURE_PATH} não foi encontrado!")
else:
    with open(SQL_PROCEDURE_PATH, "r", encoding="utf-8") as f:
        sql_procedure = f.read()

    with engine_dw.connect().execution_options(isolation_level="AUTOCOMMIT") as conn:
        dbapi_conn = conn.connection
        with dbapi_conn.cursor() as cursor:
            try:
                print("Passo 2: Criando a Stored Procedure no banco...")
                cursor.execute(sql_procedure)
                print("[OK] Stored Procedure 'dw.sp_validar_qualidade_carga' criada.")

                print("\nPasso 3: Executando o Quality Gate (Validação)...")
                cursor.execute("CALL dw.sp_validar_qualidade_carga();")
                
                # Captura os logs enviados pelo comando RAISE NOTICE do PostgreSQL
                for notice in dbapi_conn.notices:
                    print(f"Server Notice: {notice.strip()}")
                
                print("\n[SUCESSO] Dados validados. A carga da Fato está liberada para rodar!")

            except Exception as e:
                print(f"\n[BLOQUEADO] Carga interrompida pelo Quality Gate:")
                print(f"Detalhe do Erro: {e}")